# 02_generate_table1_age_stratified

This notebook creates the manuscript Table 1 summary dataset and LaTeX output for age-stratified baseline characteristics and induction practice variables.

## Purpose

The workflow converts the long-format propofol induction dataset into a patient-level analytic table, summarizes key demographic and induction variables across prespecified age strata, and exports manuscript-ready LaTeX output for inclusion in Overleaf.

## Main steps

1. Load the cleaned analysis-ready propofol induction dataset
2. Collapse medication-event rows to one row per patient or case
3. Retain patient-level demographic and induction variables used in Table 1
4. Assign patients to prespecified age strata
5. Compute descriptive summary statistics within each age stratum
6. Format summary values for manuscript presentation
7. Export LaTeX table output for direct use in Overleaf

## Main outputs

- age-stratified Table 1 summary dataset
- manuscript-ready LaTeX table output for Table 1

## Important notes

- This notebook reflects the final manuscript Table 1 workflow rather than earlier exploratory table formats.
- The final table is organized by prespecified age strata rather than a binary younger-versus-older comparison.
- Summary statistics are generated from the patient-level analytic dataset derived from the extraction and preprocessing workflow.

## Reproducibility note

This notebook is intended to document the generation of the final age-stratified baseline table used in the manuscript and associated supplementary materials.

In [ ]:
import pandas as pd
import numpy as np



In [ ]:
# Load input dataset
file_path = "propofol_analysis_ready.csv"  # replace with local path as needed
df = pd.read_csv(file_path)

In [ ]:
# Convert timestamps to datetime objects

df['ACTION_TIME'] = pd.to_datetime(df['ACTION_TIME'])


# Recover BMI when height and weight are available
bmi_recovery_mask = df['BMI'].isna() & df['WEIGHT_KG'].notna() & df['HEIGHT_IN'].notna()
df.loc[bmi_recovery_mask, 'BMI'] = df['WEIGHT_KG'] / ((df['HEIGHT_IN'] * 0.0254) ** 2)

def report_stats(df_current, step_name):
    print(f"--- {step_name} ---")
    print(f"  Rows remaining: {len(df_current)}")
    print(f"  Patients remaining: {df_current['OR_CASE_ID'].nunique()}")
    print("-" * 30)

report_stats(df, "Initial Data")

# 1. DROP PATIENTS WITH MISSING CRITICAL DATA (Sequential Attrition)
# ---------------------------------------------------------------------------
print("--- Step 1: Sequential Attrition Breakdown ---")

# A. Handle Missing SIG (Dose Signal) first
pats_missing_sig = df[df['SIG'].isna()]['OR_CASE_ID'].unique()
df = df[~df['OR_CASE_ID'].isin(pats_missing_sig)].copy()
print(f" 1. Removed for missing SIG     : {len(pats_missing_sig)} unique patients")

# B. Handle other critical variables from the remaining pool
other_vars = ['AGE', 'WEIGHT_KG', 'HEIGHT_IN', 'SEX', 'ACTION_TIME', 'UNITS', 'BMI']
pats_missing_others = df[df[other_vars].isna().any(axis=1)]['OR_CASE_ID'].unique()
df = df[~df['OR_CASE_ID'].isin(pats_missing_others)].copy()
print(f" 2. Removed for other variables : {len(pats_missing_others)} unique patients")
print("-" * 30)

# 2. SEX FILTERING (Strictly M/F)
initial_count = df['OR_CASE_ID'].nunique()
df = df[df['SEX'].str.lower().isin(['m', 'f'])].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients with invalid SEX.")

# 3. PHYSIOLOGICAL BOUNDARY FILTERS
initial_count = df['OR_CASE_ID'].nunique()
df = df[(df['HEIGHT_IN'] >= 40) & (df['HEIGHT_IN'] <= 90)].copy()
df = df[(df['BMI'] >= 8) & (df['BMI'] <= 100)].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients due to Height/BMI outliers.")

# *** FIX 2: CREATE HEIGHT_CM ***
# The simulation expects this column. If it's missing, the 'try' block fails and skips everyone.
df['HEIGHT_CM'] = df['HEIGHT_IN'] * 2.54

# 4. DOSE INTEGRITY & RECALCULATION
df['TOTAL_INFUSION_MG'] = df['TOTAL_INFUSION_MG'].fillna(0)
df['TOTAL_PROP_MG'] = df['TOTAL_BOLUS_MG'] + df['TOTAL_INFUSION_MG']

initial_count = df['OR_CASE_ID'].nunique()
df = df[(df['TOTAL_PROP_MG'] >= 20) & (df['TOTAL_PROP_MG'] <= 700)].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients with Total Dose <20 or >700 mg.")

# 5. ANTHROPOMETRIC CALCULATIONS (Devine Formula for IBW)
def calculate_ibw(row):
    height_in = row['HEIGHT_IN']
    sex = str(row['SEX']).lower()
    base_weight = 50.0 if sex == 'm' else 45.5
    over_60 = height_in - 60
    return base_weight + (2.3 * over_60)

df['IBW_DEVINE'] = df.apply(calculate_ibw, axis=1)

df['ABW_KG'] = np.where(
    df['WEIGHT_KG'] > (1.2 * df['IBW_DEVINE']),
    df['IBW_DEVINE'] + 0.4 * (df['WEIGHT_KG'] - df['IBW_DEVINE']),
    df['WEIGHT_KG']
)

# 6. CALCULATE DOSE PER KG (ABW)
df['DOSE_PER_ABW'] = df['TOTAL_PROP_MG'] / df['ABW_KG']

# 6. CALCULATE DOSE PER KG (ABW)
df['DOSE_PER_ABW'] = df['TOTAL_PROP_MG'] / df['ABW_KG']

# 7. AGE FILTER (Remove patients >= 90)
initial_count_age = df['OR_CASE_ID'].nunique()
df = df[df['AGE'] < 90].copy()
removed_age = initial_count_age - df['OR_CASE_ID'].nunique()
print(f"Removed {removed_age} patients with AGE >= 90.")

report_stats(df, "Final Cleaned Data")

In [ ]:


# Define age bins and labels
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90] # 90 is the upper bound for the 85-89 group
labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85-89']
df_analysis['Age_Group'] = pd.cut(df_analysis['AGE'], bins=bins, labels=labels, right=False)

def get_binned_stats(data):
    if len(data) == 0: return ["--"] * 11

    # Demographics
    age = f"{data['AGE'].mean():.1f} ({data['AGE'].std():.1f})"
    weight = f"{data['WEIGHT_KG'].mean():.1f} ({data['WEIGHT_KG'].std():.1f})"
    bmi = f"{data['BMI'].mean():.1f} ({data['BMI'].std():.1f})"
    n_fem = (data['SEX'].str.upper() == 'F').sum()
    sex_str = f"{n_fem:,} ({(n_fem/len(data))*100:.1f}%)"

    # ASA Status Breakdown (Vertical)
    asa_counts = data['ASA_SCORE'].value_counts().reindex([1, 2, 3, 4], fill_value=0)
    asa1 = f"{(asa_counts[1]/len(data))*100:.1f}%"
    asa2 = f"{(asa_counts[2]/len(data))*100:.1f}%"
    asa3 = f"{(asa_counts[3]/len(data))*100:.1f}%"
    asa4 = f"{(asa_counts[4]/len(data))*100:.1f}%"

    # Induction Characteristics
    n_inf = (data['HAD_INFUSION'] == 1).sum()
    inf_str = f"{n_inf:,} ({(n_inf/len(data))*100:.1f}%)"
    n_multi = (data['N_BOLUSES'] > 1).sum()
    multi_str = f"{n_multi:,} ({(n_multi/len(data))*100:.1f}%)"

    return [age, weight, bmi, sex_str, "", asa1, asa2, asa3, asa4, inf_str, multi_str]

# Labels for rows
row_labels = [
    "Age (Years)",
    "Weight (kg)",
    "Body Mass Index (kg/m$^2$)",
    "Female Sex, N (%)",
    "ASA Physical Status",
    r"\hspace{1em} ASA 1",
    r"\hspace{1em} ASA 2",
    r"\hspace{1em} ASA 3",
    r"\hspace{1em} ASA 4",
    "Infusion Administered, N (%)",
    "$>$1 Bolus During Induction, N (%)"
]

# Helper to create column names with N
def get_col_name(label, df):
    count = len(df[df['Age_Group'] == label])
    return f"{label} (N={count:,})"

# Build Split Tables with N in headers
cols_A = {get_col_name(l, df_analysis): get_binned_stats(df_analysis[df_analysis['Age_Group'] == l]) for l in labels[:4]}
cols_B = {get_col_name(l, df_analysis): get_binned_stats(df_analysis[df_analysis['Age_Group'] == l]) for l in labels[4:]}

df_multi_A = pd.DataFrame(cols_A, index=row_labels)
df_multi_B = pd.DataFrame(cols_B, index=row_labels)

# Save to .tex files for Sync
with open("table_lifespan_A.tex", "w") as f:
    f.write(df_multi_A.to_latex(escape=False, column_format='lcccc').replace('%', r'\%'))

with open("table_lifespan_B.tex", "w") as f:
    f.write(df_multi_B.to_latex(escape=False, column_format='lcccc').replace('%', r'\%'))

print("✅ Lifespan Part A and B .tex files saved with cohort Ns.")